# 面试问题：SPLADE 怎样把语义扩展写进可检索的稀疏向量？

可以直接复述的回答是：第一，SPLADE 为每个输入 token 预测整个词表的激活。第二，对正 logits 应用 `log(1+ReLU)`，再在 token 维做 max pooling。第三，结果是高维但稀疏的词项权重，可以进入倒排索引。第四，语义扩展能连接“休假→年假”等词汇错配。第五，扩展词需要 top-k、稀疏正则和敏感动作门禁。第六，应输出非零词项、倒排分项和检索指标。下面用企业制度搜索实现一个固定教学投影。

## 真实案例：员工用口语搜索 HR、财务与安全制度

知识库包含 6 条制度片段，查询集包含 5 组口语同义表达。教学编码器使用显式 expansion map 代替训练好的 MLM 权重，便于逐项观察稀疏激活；文档和查询均为脱敏样本，不代表真实企业制度。

In [1]:
documents = [  # 定义六条可读企业制度文档
    {"id": "D1", "text": "年假 余额 到期 结转"},  # 年假余额与结转制度
    {"id": "D2", "text": "差旅 报销 标准 发票"},  # 差旅费用报销制度
    {"id": "D3", "text": "工资单 下载 税款 明细"},  # 薪资材料查询制度
    {"id": "D4", "text": "离职 账号 停用 审批"},  # 离职账号处置制度
    {"id": "D5", "text": "信息 安全 规范 密码"},  # 信息安全制度
    {"id": "D6", "text": "办公 天气 城市 预警"},  # 与 HR 无关的办公辅助文档
]  # 结束六条制度文档
queries = [  # 定义五条存在词汇错配的员工查询
    {"id": "Q1", "text": "休假 还剩 多少", "expected": "D1"},  # 休假与年假、还剩与余额同义
    {"id": "Q2", "text": "出差 费用 怎么报", "expected": "D2"},  # 出差费用需要扩展到差旅报销
    {"id": "Q3", "text": "薪资 明细", "expected": "D3"},  # 薪资需要扩展到工资单
    {"id": "Q4", "text": "离职 账户 如何停用", "expected": "D4"},  # 账户与账号同义
    {"id": "Q5", "text": "安全 规则", "expected": "D5"},  # 规则与规范同义
]  # 结束五条检索评测
print("制度文档：id | text")  # 输入预览展示倒排索引的原始文本
for document in documents:  # 逐条输出六个制度片段
    print(f"{document['id']} | {document['text']}")  # 保留真实词汇供错配对照
print("口语查询：", [(query["text"], query["expected"]) for query in queries])  # 展示同义表达和期望文档


制度文档：id | text
D1 | 年假 余额 到期 结转
D2 | 差旅 报销 标准 发票
D3 | 工资单 下载 税款 明细
D4 | 离职 账号 停用 审批
D5 | 信息 安全 规范 密码
D6 | 办公 天气 城市 预警
口语查询： [('休假 还剩 多少', 'D1'), ('出差 费用 怎么报', 'D2'), ('薪资 明细', 'D3'), ('离职 账户 如何停用', 'D4'), ('安全 规则', 'D5')]


## Baseline / 基线：只按原词重合排序

精确词项检索无法理解休假与年假、出差与差旅等关系。先展示 Q1 在六篇文档上的原词重合数。

In [2]:
def lexical_score(query_text, document_text):  # 计算空格分词后的精确词项重合
    return len(set(query_text.split()) & set(document_text.split()))  # 每个命中词贡献一分
baseline_q1 = sorted([(lexical_score(queries[0]["text"], document["text"]), document["id"]) for document in documents], reverse=True)  # 对休假查询运行词项基线
print("Q1 词项基线：score | document")  # 输出所有文档的精确匹配结果
for score, document_id in baseline_q1:  # 逐文档展示重合数
    print(f"{score} | {document_id}")  # 观察同义词导致所有文档零分或平分


Q1 词项基线：score | document
0 | D6
0 | D5
0 | D4
0 | D3
0 | D2
0 | D1


## 核心实现：词表 logits、非线性与 max pooling

每个输入 token 激活自身词项，并通过教学投影激活同义词。使用与 SPLADE 一致的 `log1p(ReLU)` 和 token 维 max pooling 生成稀疏向量。

In [3]:
import torch  # 使用 PyTorch 张量实现词表激活与 max pooling
vocabulary = sorted({token for document in documents for token in document["text"].split()} | {token for query in queries for token in query["text"].split()})  # 构建文档和查询共享词表
term_to_id = {term: index for index, term in enumerate(vocabulary)}  # 建立词项到稀疏向量坐标的映射
expansions = {"休假": {"年假": 2.8}, "还剩": {"余额": 2.6}, "出差": {"差旅": 2.8}, "费用": {"报销": 2.5}, "薪资": {"工资单": 2.9}, "账户": {"账号": 2.7}, "规则": {"规范": 2.6}}  # 定义可审计同义词投影
def splade_encode(text, top_k=6, blocked_terms=None):  # 生成经过稀疏裁剪的词表权重
    tokens = text.split()  # 使用显式空格分词保留教学可读性
    token_logits = torch.zeros(len(tokens), len(vocabulary))  # 初始化每个输入 token 对整个词表的 logits
    for row, token in enumerate(tokens):  # 逐 token 写入自身和同义扩展激活
        token_logits[row, term_to_id[token]] = 3.0  # 原始词项获得最高自身激活
        for expanded, value in expansions.get(token, {}).items():  # 遍历当前 token 的语义扩展
            token_logits[row, term_to_id[expanded]] = value  # 把同义概念写入词表对应坐标
    weights = torch.log1p(torch.relu(token_logits)).max(dim=0).values  # 应用非线性并在 token 维做 max pooling
    blocked = blocked_terms or set()  # 获取当前查询不允许自动扩展的敏感词集合
    for term in blocked:  # 对敏感动作词应用确定性门禁
        if term in term_to_id and term not in tokens:  # 用户未显式输入的敏感词不能由模型扩展产生
            weights[term_to_id[term]] = 0.0  # 清除敏感扩展权重
    if int((weights > 0).sum()) > top_k:  # 非零词项超过索引预算时只保留 Top-k
        threshold = torch.topk(weights, top_k).values[-1]  # 计算第 k 大稀疏权重
        weights = torch.where(weights >= threshold, weights, torch.zeros_like(weights))  # 裁剪低权重扩展
    return weights  # 返回可写入倒排索引的稀疏向量
query_vector = splade_encode(queries[0]["text"])  # 编码“休假 还剩 多少”查询
query_terms = [(term, float(query_vector[index])) for term, index in term_to_id.items() if query_vector[index] > 0]  # 提取非零扩展词和权重
print("Q1 SPLADE 非零词项：", [(term, round(weight, 3)) for term, weight in query_terms])  # 展示休假→年假和还剩→余额扩展
print(f"词表维度={len(vocabulary)}，Q1 非零维度={len(query_terms)}")  # 展示高维稀疏表示的结构


Q1 SPLADE 非零词项： [('休假', 1.386), ('余额', 1.281), ('多少', 1.386), ('年假', 1.335), ('还剩', 1.386)]
词表维度=34，Q1 非零维度=5


## 失败案例与修正：敏感动作不能仅凭语义扩展触发

若模型把“离职咨询”扩展成“账号删除”，检索结果可用于阅读，但不能直接转成删除工具调用。这里演示敏感扩展清除，并把检索与动作授权分离。

In [4]:
dangerous_expansions = dict(expansions)  # 复制正常同义词表以构造危险扩展
dangerous_expansions["离职"] = {"账号": 2.5, "停用": 2.4}  # 模拟模型从主题词扩展出账户处置动作
original_expansions = expansions  # 保存正常投影以便恢复
expansions = dangerous_expansions  # 临时启用包含敏感动作的投影
unsafe_vector = splade_encode("离职 规则", blocked_terms=set())  # 不做敏感门禁时生成稀疏查询
safe_vector = splade_encode("离职 规则", blocked_terms={"账号", "停用"})  # 清除用户未显式输入的敏感动作词
expansions = original_expansions  # 恢复正常教学投影供后续评测
unsafe_terms = [term for term, index in term_to_id.items() if unsafe_vector[index] > 0]  # 提取门禁前非零词项
safe_terms = [term for term, index in term_to_id.items() if safe_vector[index] > 0]  # 提取门禁后非零词项
action_authorized = "停用" in "离职 规则".split() and "账号" in "离职 规则".split()  # 只有用户显式表达动作和对象才可能进入授权流程
print("敏感扩展修正前：", unsafe_terms)  # 展示主题词如何产生账号停用概念
print("敏感扩展修正后：", safe_terms)  # 展示高风险动作词被查询门禁移除
print("是否允许调用账号停用工具：", action_authorized)  # 明确检索相关性不能替代动作授权


敏感扩展修正前： ['停用', '离职', '规则', '规范', '账号']
敏感扩展修正后： ['离职', '规则', '规范']
是否允许调用账号停用工具： False


## 结果表：五条查询的 Lexical 与 SPLADE Recall@1

In [5]:
document_vectors = {document["id"]: splade_encode(document["text"]) for document in documents}  # 预计算六篇文档的稀疏索引向量
lexical_hits = 0  # 初始化精确词项首位命中数
splade_hits = 0  # 初始化稀疏扩展首位命中数
print("query | expected | lexical_top1 | splade_top1 | top_contributions")  # 输出逐查询检索与分项证据
for query in queries:  # 在同一批五条口语查询上比较两种方案
    lexical_top = max(documents, key=lambda document: (lexical_score(query["text"], document["text"]), document["id"]))["id"]  # 获取精确词项基线首位文档
    encoded_query = splade_encode(query["text"])  # 生成当前查询的稀疏扩展向量
    sparse_scores = {document_id: float((encoded_query * vector).sum()) for document_id, vector in document_vectors.items()}  # 计算倒排点积等价分数
    splade_top = max(sparse_scores, key=sparse_scores.get)  # 获取最高稀疏相关性文档
    contributions = [(term, round(float(encoded_query[index] * document_vectors[splade_top][index]), 3)) for term, index in term_to_id.items() if encoded_query[index] * document_vectors[splade_top][index] > 0]  # 提取命中词项分数贡献
    lexical_hits += int(lexical_top == query["expected"])  # 累加词项基线正确数
    splade_hits += int(splade_top == query["expected"])  # 累加稀疏扩展正确数
    print(f"{query['text']} | {query['expected']} | {lexical_top} | {splade_top} | {contributions}")  # 展示同义扩展如何支持首位结果
lexical_recall = lexical_hits / len(queries)  # 计算精确词项 Recall@1
splade_recall = splade_hits / len(queries)  # 计算 SPLADE Recall@1
print(f"Recall@1：lexical={lexical_recall:.1%}，SPLADE={splade_recall:.1%}")  # 输出同评测集汇总


query | expected | lexical_top1 | splade_top1 | top_contributions
休假 还剩 多少 | D1 | D6 | D1 | [('余额', 1.776), ('年假', 1.851)]
出差 费用 怎么报 | D2 | D6 | D2 | [('差旅', 1.851), ('报销', 1.737)]
薪资 明细 | D3 | D3 | D3 | [('工资单', 1.887), ('明细', 1.922)]
离职 账户 如何停用 | D4 | D4 | D4 | [('离职', 1.922), ('账号', 1.814)]
安全 规则 | D5 | D5 | D5 | [('安全', 1.922), ('规范', 1.776)]
Recall@1：lexical=60.0%，SPLADE=100.0%


## 结果解读

Q1 的非零向量同时保留原词“休假、还剩、多少”，并激活“年假、余额”，因此可以通过倒排点积命中 D1。五条查询的贡献表展示了哪组原词或扩展词真正影响排名。敏感扩展案例说明 SPLADE 只解决召回，不能把语义相似直接解释为副作用授权。

## 生产边界

真实 SPLADE 需要 MLM 编码器训练、FLOPS 稀疏正则、词表版本、文档离线编码和高效倒排引擎。Top-k 会影响召回与索引大小，中文 subword 还需还原可解释词项。安全系统必须把搜索候选与工具调用权限分层。本例的 expansion map 是教学替身，不是学习结果。

## 最小回归测试

In [6]:
assert len(documents) >= 5 and len(queries) >= 5  # 保证稀疏检索案例包含足够文档与查询
assert "年假" in {term for term, _ in query_terms} and "余额" in {term for term, _ in query_terms}  # 保证 Q1 产生两个关键同义扩展
assert "账号" in unsafe_terms and "账号" not in safe_terms  # 保证敏感动作门禁清除非显式账号扩展
assert action_authorized is False  # 保证检索扩展不能自动授权账号停用工具
assert splade_recall > lexical_recall  # 保证语义稀疏扩展在同一口语查询集上优于原词匹配
assert splade_recall >= 0.8  # 保证五条教学查询的大多数首位结果正确
